# TC-WPN — Stage B: Phase A validation run (GPU)

**Accelerator: GPU T4.** This is the single-seed, single-K run your supervisor
asked for before you spend quota on the full grid:

> TF-IDF, ClinicalBERT probe, ProtoNet, TC-WPN — K=5, seed 42, and nothing else.

Do not expand this to 6 models × 4 K × 5 seeds until the numbers below have been
read. If the task is mis-specified, a large grid just produces a large pile of
wrong results.

### Kaggle quota notes
- Weekly GPU quota is around 30 h and floats with demand; check the figure in
  the right-hand pane before starting.
- A session runs up to 12 h but disconnects after ~90 min idle, so use
  **Save & Run All (Commit)** rather than babysitting an interactive session.
- You may hold 1 interactive GPU session plus 2 commit sessions. That means
  ProtoNet and TC-WPN can train as two parallel commits if you split them into
  separate notebook versions.

### Before you run
Add the Stage A output as an input dataset.

In [ ]:
# ---------------------------------------------------------------------------
# 1. Repo + dependencies
# ---------------------------------------------------------------------------
!rm -rf /kaggle/working/tcwpn_test
!git clone -q https://github.com/dulhara79/tcwpn_test.git /kaggle/working/tcwpn_test
%cd /kaggle/working/tcwpn_test
!pip install -q -r requirements.txt 2>&1 | tail -2

import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
# ---------------------------------------------------------------------------
# 2. Locate the Stage A artefacts
# ---------------------------------------------------------------------------
from pathlib import Path

STAGE_A = None
for d in sorted(Path("/kaggle/input").glob("*")):
    if (d / "data" / "clean" / "pkl").exists():
        STAGE_A = d / "data" / "clean"
        break
if STAGE_A is None:
    raise SystemExit("Stage A output not found. Add it via + Add Input.")

PKL_DIR  = str(STAGE_A / "pkl")
PLAN_DIR = str(STAGE_A / "plans")
STEM     = "psych_mimic4idx"     # must match the Stage A cohort filename
RESULTS  = "/kaggle/working/results"
K        = 5
SEED     = 42

print("pkl :", PKL_DIR)
print("plan:", PLAN_DIR)
!ls {PKL_DIR}
!ls {PLAN_DIR} | head

## 3. Shallow baselines first — and read them before training anything

Both are fitted **inside each episode from the K support examples only**, so
they are genuine few-shot baselines rather than fully-supervised classifiers
wearing a few-shot label.

**This is the decision point.** If TF-IDF is at or above the neural models, that
is the headline result and the paper changes shape. Do not proceed to the GPU
training cells on autopilot.

In [ ]:
!python -m scripts.run_shallow_baselines \
    --stem {STEM} --k {K} --seed {SEED} --baseline tfidf_lr --split test \
    --pkl-dir {PKL_DIR} --plan-dir {PLAN_DIR} --results {RESULTS}

In [ ]:
!python -m scripts.run_shallow_baselines \
    --stem {STEM} --k {K} --seed {SEED} --baseline bert_probe --split test \
    --pkl-dir {PKL_DIR} --plan-dir {PLAN_DIR} --results {RESULTS}

## 4. Train ProtoNet, then TC-WPN

Same code path, same frozen episodes, differing only by the ablation switches in
the config. The threshold is locked on validation inside `train.py` and is never
re-tuned on test.

In [ ]:
!python -m scripts.train --config configs/protonet.yaml \
    --k {K} --seed {SEED} --stem {STEM} \
    --pkl-dir {PKL_DIR} --plan-dir {PLAN_DIR} --results {RESULTS}

In [ ]:
!python -m scripts.train --config configs/tcwpn_full.yaml \
    --k {K} --seed {SEED} --stem {STEM} \
    --pkl-dir {PKL_DIR} --plan-dir {PLAN_DIR} --results {RESULTS}

## 5. Evaluate on test

Patient-level metrics with patient bootstrap. Every metric your supervisor asked
for — AUROC, PR-AUC, F1, sensitivity, specificity, Brier, ECE — comes out of
here into `eval_test.json`, plus `predictions_test.csv` which is what the paired
DeLong test consumes.

In [ ]:
PROTO = f"{RESULTS}/{STEM}/protonet_k{K}_seed{SEED}"
TCWPN = f"{RESULTS}/{STEM}/tcwpn_full_k{K}_seed{SEED}"

!python -m scripts.evaluate --run {PROTO} --split test \
    --pkl-dir {PKL_DIR} --plan-dir {PLAN_DIR} --bootstrap 2000
!python -m scripts.evaluate --run {TCWPN} --split test \
    --pkl-dir {PKL_DIR} --plan-dir {PLAN_DIR} --bootstrap 2000

## 6. Robustness: the same patients, blinded text

Identical patients, identical episode plan, only the characters differ. The drop
from this comparison is attributable to the lexical shortcut — which the
archived blinded-vs-unblinded comparison could not claim, because those two arms
ran over different record sets at different prevalences.

In [ ]:
!python -m scripts.evaluate --run {TCWPN} --split test --blind dx_meds \
    --pkl-dir {PKL_DIR} --plan-dir {PLAN_DIR} --bootstrap 2000
!python -m scripts.evaluate --run {PROTO} --split test --blind dx_meds \
    --pkl-dir {PKL_DIR} --plan-dir {PLAN_DIR} --bootstrap 2000

## 7. Comparison table and the paired significance test

`pair` runs DeLong over the two patient-level score vectors. Report the p-value
it returns whichever way it points. With a small gap and overlapping intervals,
a non-significant result is the honest finding and is perfectly reportable.

In [ ]:
!python -m scripts.compare_models table --results {RESULTS}/{STEM} --split test \
    --out /kaggle/working/results_table.csv

In [ ]:
!python -m scripts.compare_models pair \
    --a {PROTO}/predictions_test.csv \
    --b {TCWPN}/predictions_test.csv \
    --out /kaggle/working/delong_protonet_vs_tcwpn.json

## 8. The 13 numbers to send your supervisor

Everything requested in point 19 of the feedback, in one block.

In [ ]:
import json, glob, os
import pandas as pd

def show(path, title):
    if not os.path.exists(path):
        print(f"[missing] {title}: {path}"); return None
    d = json.load(open(path))
    print("="*70); print(title)
    print(json.dumps(d, indent=2)[:2500])
    return d

# 1-3  cohort audit, split summary, leakage certificate
for f in sorted(glob.glob(str(STAGE_A / "audit_*.json"))):
    show(f, f"COHORT AUDIT — {os.path.basename(f)}")
for f in sorted(glob.glob(str(STAGE_A / "*_index_report.json"))):
    show(f, f"INDEX-TIME REPORT — {os.path.basename(f)}")
for f in sorted(glob.glob(str(STAGE_A / "plans" / "leakage_certificate_*.json"))):
    show(f, f"LEAKAGE CERTIFICATE — {os.path.basename(f)}")

# 4-13  the K=5 results
rows = []
for run in sorted(glob.glob(f"{RESULTS}/{STEM}/*")):
    ev = os.path.join(run, "eval_test.json")
    mf = os.path.join(run, "manifest.json")
    if not os.path.exists(ev):
        continue
    m = json.load(open(ev))["metrics"]
    thr = json.load(open(mf)).get("locked_threshold") if os.path.exists(mf) else None
    m["run"] = os.path.basename(run)
    m["locked_threshold"] = thr
    rows.append(m)

if rows:
    df = pd.DataFrame(rows).set_index("run")
    cols = [c for c in ["auroc", "auroc_ci_low", "auroc_ci_high", "pr_auc",
                        "f1", "sensitivity", "specificity", "brier", "ece",
                        "n_patients", "prevalence", "locked_threshold"]
            if c in df.columns]
    print("\n" + "="*70)
    print("PHASE A SUMMARY — K=5, seed 42, test split")
    print("="*70)
    print(df[cols].to_string())
    df.to_csv("/kaggle/working/phase_a_summary.csv")
else:
    print("No eval_test.json found — did the training cells complete?")

## What to do with the result

If AUROC lands well below the archived 0.97–0.98, that is the expected
consequence of removing the shortcut, not a failure. The question that matters
is the **gap** between TC-WPN and the baselines on the clean task, and whether
DeLong says it is real.

Before expanding to five seeds, send your supervisor the summary above and the
three JSON artefacts. A grid built on a mis-specified task is expensive in a way
Kaggle quota makes painful.